# LightGBM-Huber — `rv_plus_turnover` on R1000/R2000 daily

One model, one config cell. The original notebook trained seven feature sets and branched on
whether `TRET_T1D` was a past or future return; both are gone. `TRET_T1D` is **confirmed to be
the return ending at `day = t`**, so it is a feature and the label comes from the next market
date.

## Features — 9, all known at the close of day $t$

$$\underbrace{r_{1d},\ r_{5d},\ r_{21d},\ r_{63d},\ r_{126d},\ r_{252d}}_{\text{compounded returns}}
,\quad \underbrace{\sigma_{21d},\ \sigma_{63d}}_{\text{population vol, ddof}=0},\quad
\text{turnover}_{21d}$$

No price and no market cap — that is what makes this `rv_plus_turnover` rather than the
eleven-feature model. Both columns are still loaded, but only to describe *what the model
selects*, never to fit it.

## Label

$$y_t = \texttt{TRET\_T1D}(t+1)\quad\text{— the return from } t \text{ to } t+1$$

assigned **only when $t$ and $t+1$ are adjacent global market dates**. A security absent on the
next trading day gets no label rather than one silently borrowed from further ahead. The same
adjacency rule guards every rolling window: a window that spans a gap is `NaN`, not a stale
value carried across it.

## No universe screen

No volume, turnover, price or size filter. Zero-volume names stay in. That is deliberate — the
selection-characteristics table at the end exists to show whether the model drifts into them.


## 1. Configuration — everything you would want to change lives here

In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field, asdict
from pathlib import Path
import math, json, platform, warnings

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import lightgbm as lgb
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 170)


@dataclass
class Config:
    # ---- data -------------------------------------------------------------------------
    data_path: Path = Path("manager_holdings/R1000_R2000_daily_turnover21D.parquet")
    output_dir: Path = Path("lightgbm_rv_turnover_results")

    # ---- THE SPLIT ---------------------------------------------------------------------
    # mode = "single"        one fixed train/valid/test, set by the six dates below
    #        "walk_forward"  refit repeatedly, rolling the whole thing forward. Every test
    #                        block is scored by a model that never saw it, and the blocks are
    #                        concatenated into one continuous out-of-sample series.
    mode: str = "walk_forward"

    # -- single mode. test_end = None means "to the end of the data".
    train_start: str | None = None
    train_end:   str = "2022-12-31"
    valid_start: str = "2023-01-01"
    valid_end:   str = "2023-12-31"
    test_start:  str = "2024-01-01"
    test_end:    str | None = None

    # -- walk-forward mode, all in months
    wf_train_months: int = 36
    wf_valid_months: int = 12
    wf_test_months:  int = 12
    wf_step_months:  int | None = None   # None = wf_test_months, so blocks tile exactly
    wf_first_test:   str | None = None   # None = as early as the history allows
    wf_last_test:    str | None = None   # None = to the end of the data
    wf_expanding:    bool = False        # True: train start pinned to the data start

    # ---- portfolio ----------------------------------------------------------------------
    top_k: int = 10                 # names in each leg of the long/short
    hac_lags: int = 5               # Newey-West lags for the daily-mean t-stat

    # ---- LightGBM ------------------------------------------------------------------------
    n_estimators: int = 500
    early_stopping_rounds: int = 30
    learning_rate: float = 0.05
    num_leaves: int = 63
    huber_alpha: float = 0.9
    subsample: float = 0.8
    colsample_bytree: float = 0.9
    reg_lambda: float = 1.0
    max_bin: int = 127
    seed: int = 1337
    n_jobs: int = -1

    # ---- outputs -------------------------------------------------------------------------
    save_model: bool = False
    save_row_predictions: bool = False

    def dates(self):
        f = lambda s: None if s is None else pd.Timestamp(s)
        return (f(self.train_start), f(self.train_end), f(self.valid_start),
                f(self.valid_end), f(self.test_start), f(self.test_end))

    def validate(self):
        if self.mode not in {"single", "walk_forward"}:
            raise ValueError("mode must be 'single' or 'walk_forward'")
        if self.mode == "walk_forward":
            if min(self.wf_train_months, self.wf_valid_months, self.wf_test_months) < 1:
                raise ValueError("walk-forward window lengths must be >= 1 month")
            step = self.wf_step_months or self.wf_test_months
            if step > self.wf_test_months:
                raise ValueError("wf_step_months > wf_test_months leaves untested gaps")
            if step < self.wf_test_months:
                raise ValueError("wf_step_months < wf_test_months makes test blocks OVERLAP, "
                                 "so days are scored twice and t-stats are inflated")
            return self
        a, b, c, d, e, g = self.dates()
        if a is not None and a >= b:
            raise ValueError("train_start must precede train_end")
        if not (b < c <= d < e):
            raise ValueError("need train_end < valid_start <= valid_end < test_start")
        if g is not None and e > g:
            raise ValueError("test_end must be on or after test_start")
        if min(self.top_k, self.n_estimators, self.early_stopping_rounds) < 1:
            raise ValueError("top_k, n_estimators, early_stopping_rounds must be positive")
        return self


CFG = Config().validate()

FEATURES = [
    "return_1d", "return_5d", "return_21d", "return_63d", "return_126d", "return_252d",
    "volatility_21d", "volatility_63d",
    "turnover_21day",
]
# loaded for diagnostics only -- never fitted on
DIAGNOSTIC_COLUMNS = ["DOLLARVOLUME_AVG21D", "market_cap"]

_desc = ({"mode": "single",
          "train": f"{CFG.train_start or 'start'} .. {CFG.train_end}",
          "validation": f"{CFG.valid_start} .. {CFG.valid_end}",
          "test": f"{CFG.test_start} .. {CFG.test_end or 'end'}"}
         if CFG.mode == "single" else
         {"mode": "walk_forward",
          "train": f"{CFG.wf_train_months}m"
                   + (" (expanding)" if CFG.wf_expanding else " (rolling)"),
          "validation": f"{CFG.wf_valid_months}m",
          "test": f"{CFG.wf_test_months}m every "
                  f"{CFG.wf_step_months or CFG.wf_test_months}m"})
display(pd.Series({"data_path": str(CFG.data_path), **_desc,
                   "features": len(FEATURES), "top_k": CFG.top_k},
                  name="value").to_frame())


## 2. Load and validate

The load fails loudly on a missing column, an unparseable date, or a duplicated
`(day, security)` — all three would corrupt the alignment silently otherwise.

In [ ]:
REQUIRED = ["day", "security", "TRET_T1D", "turnover_21day"] + DIAGNOSTIC_COLUMNS

if not CFG.data_path.exists():
    raise FileNotFoundError(f"not found: {CFG.data_path.resolve()}")
schema = list(pq.ParquetFile(CFG.data_path).schema.names)
missing = sorted(set(REQUIRED) - set(schema))
if missing:
    raise ValueError(f"parquet is missing required columns: {missing}")

frame = pd.read_parquet(CFG.data_path, columns=REQUIRED).copy()
frame["day"] = pd.to_datetime(frame["day"], errors="coerce").dt.normalize()
if frame["day"].isna().any():
    raise ValueError(f"{int(frame['day'].isna().sum()):,} unparseable dates")
if frame["security"].isna().any():
    raise ValueError(f"{int(frame['security'].isna().sum()):,} missing security ids")
frame["security"] = frame["security"].astype("string")
if frame.duplicated(["day", "security"]).any():
    raise ValueError("duplicated (day, security) rows -- alignment would be undefined")

frame = frame.sort_values(["security", "day"]).reset_index(drop=True)
print(f"{len(frame):,} rows | {frame.security.nunique():,} securities | "
      f"{frame.day.nunique():,} market dates | "
      f"{frame.day.min().date()} to {frame.day.max().date()}")


## 3. Align the label, then build the rolling features

`TRET_T1D(t)` is the return **ending** at $t$, so it is the one-day feature. The label is the
same column one market date later.

Every construction below is guarded by **market-date adjacency**: a value is used only when the
observations really are consecutive trading days for that security. Without it, a name that
stops trading for a month would have its 21-day window quietly span that gap.

In [ ]:
market_dates = pd.Index(frame["day"].drop_duplicates().sort_values())
date_number = pd.Series(np.arange(len(market_dates)), index=market_dates)
frame["_dn"] = frame["day"].map(date_number).astype(int)

raw = frame["TRET_T1D"].to_numpy(dtype=float)
ret_1d = np.full(len(frame), np.nan)
target = np.full(len(frame), np.nan)

for pos in frame.groupby("security", sort=False).indices.values():
    pos = np.asarray(pos, dtype=int)
    dn = frame["_dn"].to_numpy()[pos]
    v = raw[pos]
    adjacent = np.diff(dn) == 1          # t and t+1 are consecutive market dates
    ret_1d[pos] = v                      # TRET_T1D(t) ends at t -> a feature
    target[pos[:-1][adjacent]] = v[1:][adjacent]   # label = TRET_T1D(t+1)

frame["return_1d"] = ret_1d
frame["target_return"] = target


def rolling_compound(values, dn, window):
    """Compounded return over `window` consecutive market dates, or NaN."""
    out = np.full(len(values), np.nan)
    if len(values) < window:
        return out
    factors = 1.0 + values
    valid = np.isfinite(values) & (factors >= 0.0)
    zeros = valid & (factors == 0.0)
    logs = np.zeros(len(values))
    pos = valid & (factors > 0.0)
    logs[pos] = np.log(factors[pos])
    bad_pref = np.r_[0, np.cumsum(~valid)]
    zero_pref = np.r_[0, np.cumsum(zeros)]
    log_pref = np.r_[0.0, np.cumsum(logs)]
    end = np.arange(window - 1, len(values)); start = end - window + 1
    usable = ((bad_pref[end + 1] - bad_pref[start] == 0)
              & (dn[end] - dn[start] == window - 1))          # no gap inside the window
    nz = zero_pref[end + 1] - zero_pref[start]
    out[end[usable & (nz > 0)]] = -1.0                        # a -100% day wipes the window
    normal = usable & (nz == 0)
    with np.errstate(over="ignore", invalid="ignore"):
        out[end[normal]] = np.expm1((log_pref[end + 1] - log_pref[start])[normal])
    return out


def rolling_vol(values, dn, window):
    """Population sd (ddof=0) over `window` consecutive market dates, or NaN."""
    out = np.full(len(values), np.nan)
    if len(values) < window:
        return out
    valid = np.isfinite(values)
    safe = np.where(valid, values, 0.0)
    c = np.r_[0, np.cumsum(valid)]
    s = np.r_[0.0, np.cumsum(safe)]
    q = np.r_[0.0, np.cumsum(safe * safe)]
    end = np.arange(window - 1, len(values)); start = end - window + 1
    usable = ((c[end + 1] - c[start] == window) & (dn[end] - dn[start] == window - 1))
    tot = s[end + 1] - s[start]; tot2 = q[end + 1] - q[start]
    var = np.maximum(tot2 / window - (tot / window) ** 2, 0.0)
    out[end[usable]] = np.sqrt(var[usable])
    return out


RETURN_WINDOWS = (5, 21, 63, 126, 252)
VOL_WINDOWS = (21, 63)
for w in RETURN_WINDOWS:
    frame[f"return_{w}d"] = np.nan
for w in VOL_WINDOWS:
    frame[f"volatility_{w}d"] = np.nan

for pos in frame.groupby("security", sort=False).indices.values():
    pos = np.asarray(pos, dtype=int)
    v = frame["return_1d"].to_numpy()[pos]
    dn = frame["_dn"].to_numpy()[pos]
    for w in RETURN_WINDOWS:
        frame.loc[pos, f"return_{w}d"] = rolling_compound(v, dn, w)
    for w in VOL_WINDOWS:
        frame.loc[pos, f"volatility_{w}d"] = rolling_vol(v, dn, w)

miss = [c for c in FEATURES if c not in frame.columns]
if miss:
    raise ValueError(f"features not built: {miss}")
print("features built:", FEATURES)


In [ ]:
# ---- Alignment audit. Read this before trusting anything below. --------------------
# TRET_T1D(t) must equal return_1d(t), and must equal target_return(t-1) wherever t-1 and t
# are adjacent market dates.
sec = frame.groupby("security", sort=False).size().idxmax()
display(frame.loc[frame.security == sec,
                  ["day", "security", "TRET_T1D", "return_1d", "target_return"]].head(8))

same = np.isclose(frame["TRET_T1D"], frame["return_1d"], equal_nan=True)
print(f"TRET_T1D == return_1d : {same.mean():.4%} of rows   (must be 100%)")

# A row is unlabelled exactly when the NEXT market date is not the security's next row --
# the last row of each security, and any row sitting on the near side of a trading gap.
gap_next = frame.groupby("security", sort=False)["_dn"].shift(-1) - frame["_dn"] != 1
unlab = frame["target_return"].isna()
print(f"unlabelled rows       : {unlab.sum():,} ({unlab.mean():.2%})")
print(f"  of which explained by a gap or a series end: {(unlab & gap_next).sum():,}")
print(f"  unexplained (should be 0)                  : {(unlab & ~gap_next).sum():,}")

# And the rolling windows must NOT span a gap: the first rows after a break carry NaN.
resume = frame.groupby("security", sort=False)["_dn"].diff() > 1
if resume.any():
    i = np.flatnonzero(resume.to_numpy())[0]
    print(f"\nfirst row after a trading gap ({frame.security.iat[i]}, "
          f"{frame.day.iat[i].date()}) -- windows must be NaN here:")
    display(frame.iloc[[i]][["day", "security", "return_5d", "return_21d",
                             "volatility_21d", "volatility_63d"]])
else:
    print("\nno trading gaps in this data -- the adjacency guards never bind")


## 4. Folds

`mode="single"` gives one fixed train/valid/test. `mode="walk_forward"` refits repeatedly,
sliding the whole thing forward:

```
fold 1  [------ train 36m ------][- valid 12m -][- test 12m -]
fold 2                  [------ train 36m ------][- valid 12m -][- test 12m -]
fold 3                                  [------ train 36m ------][- valid 12m -][- test 12m -]
```

Every test block is scored by a model fitted only on data **before** it, and the blocks are
concatenated into one continuous out-of-sample series. `wf_step_months` is forced to equal
`wf_test_months`: a smaller step would score the same day in two folds and inflate the
t-stats, a larger one would leave days untested.

`wf_expanding=True` pins the training start to the beginning of the data instead of rolling
it, so the model sees more history in later folds.


In [ ]:
def make_folds(frame: pd.DataFrame, cfg: Config) -> list[dict]:
    """Fold boundaries as timestamps. Nothing is trained here."""
    lo, hi = frame["day"].min(), frame["day"].max()
    if cfg.mode == "single":
        ts, te_, vs, ve, tts, tte = cfg.dates()
        return [{"fold": 0, "train_start": ts or lo, "train_end": te_,
                 "valid_start": vs, "valid_end": ve,
                 "test_start": tts, "test_end": tte or hi}]

    mo = pd.DateOffset(months=1)
    step = cfg.wf_step_months or cfg.wf_test_months
    # earliest test start that still leaves room for train + valid behind it
    earliest = lo + mo * (cfg.wf_train_months + cfg.wf_valid_months)
    first = max(earliest, pd.Timestamp(cfg.wf_first_test)) if cfg.wf_first_test else earliest
    last = pd.Timestamp(cfg.wf_last_test) if cfg.wf_last_test else hi

    folds, k, cursor = [], 0, first
    while cursor <= last:
        test_end = min(cursor + mo * cfg.wf_test_months - pd.Timedelta(days=1), last)
        valid_end = cursor - pd.Timedelta(days=1)
        valid_start = cursor - mo * cfg.wf_valid_months
        train_end = valid_start - pd.Timedelta(days=1)
        train_start = lo if cfg.wf_expanding else train_end - mo * cfg.wf_train_months
        if train_start < lo:
            train_start = lo
        if train_end > train_start and test_end > cursor:
            folds.append({"fold": k, "train_start": train_start, "train_end": train_end,
                          "valid_start": valid_start, "valid_end": valid_end,
                          "test_start": cursor, "test_end": test_end})
            k += 1
        cursor = cursor + mo * step
    if not folds:
        raise ValueError("no folds -- the data is shorter than train+valid+test")
    return folds


def split_fold(frame: pd.DataFrame, f: dict, cfg: Config):
    """Rows for one fold. Unlabelled rows are dropped: they can be neither fitted nor scored."""
    lab = frame["target_return"].notna()
    tr = frame.loc[lab & frame.day.between(f["train_start"], f["train_end"])]
    va = frame.loc[lab & frame.day.between(f["valid_start"], f["valid_end"])]
    te = frame.loc[lab & frame.day.between(f["test_start"], f["test_end"])]
    return tr.copy(), va.copy(), te.copy()


FOLDS = make_folds(frame, CFG)
rows = []
for f in FOLDS:
    tr, va, te = split_fold(frame, f, CFG)
    rows.append({"fold": f["fold"],
                 "train": f"{f['train_start'].date()}..{f['train_end'].date()}",
                 "valid": f"{f['valid_start'].date()}..{f['valid_end'].date()}",
                 "test":  f"{f['test_start'].date()}..{f['test_end'].date()}",
                 "train_rows": len(tr), "valid_rows": len(va), "test_rows": len(te),
                 "test_days": te.day.nunique()})
fold_table = pd.DataFrame(rows)
display(fold_table)

bad = fold_table[(fold_table.train_rows < 1000) | (fold_table.test_days < 5)]
if len(bad):
    print("[warn] these folds are thin; consider longer windows or fewer folds:")
    display(bad)

# the test blocks must tile without overlap -- otherwise a day is scored twice
spans = [(f["test_start"], f["test_end"]) for f in FOLDS]
overlap = any(spans[i][1] >= spans[i + 1][0] for i in range(len(spans) - 1))
print(f"{len(FOLDS)} fold(s) | test blocks overlap: {overlap}  (must be False)")


## 5. Train — one model per fold

In [ ]:
def fit_one(tr: pd.DataFrame, va: pd.DataFrame) -> lgb.LGBMRegressor:
    m = lgb.LGBMRegressor(
        objective="huber", alpha=CFG.huber_alpha,
        n_estimators=CFG.n_estimators, learning_rate=CFG.learning_rate,
        num_leaves=CFG.num_leaves,
        min_child_samples=min(200, max(1, len(tr) // 100)),
        subsample=CFG.subsample, subsample_freq=1,
        colsample_bytree=CFG.colsample_bytree, reg_lambda=CFG.reg_lambda,
        max_bin=CFG.max_bin, random_state=CFG.seed, n_jobs=CFG.n_jobs,
        verbosity=-1, deterministic=True, force_col_wise=True,
    )
    m.fit(tr[FEATURES].astype("float32"), tr["target_return"].astype("float32"),
          eval_set=[(va[FEATURES].astype("float32"),
                     va["target_return"].astype("float32"))],
          eval_metric="l2",
          callbacks=[lgb.early_stopping(CFG.early_stopping_rounds, verbose=False)])
    return m


models, scored_parts, imp_parts, fit_rows = {}, [], [], []
for f in FOLDS:
    tr, va, te = split_fold(frame, f, CFG)
    if len(tr) < 100 or va.empty or te.empty:
        print(f"fold {f['fold']}: skipped (train {len(tr)}, valid {len(va)}, test {len(te)})")
        continue
    m = fit_one(tr, va)
    best = int(m.best_iteration_ or m.n_estimators)
    models[f["fold"]] = m

    s = te[["day", "security", "target_return", *FEATURES, *DIAGNOSTIC_COLUMNS]].copy()
    s["score"] = m.predict(te[FEATURES].astype("float32"), num_iteration=best)
    s["fold"] = f["fold"]
    if not np.isfinite(s["score"]).all():
        raise ValueError(f"fold {f['fold']} produced non-finite predictions")
    scored_parts.append(s)

    gain = m.booster_.feature_importance(importance_type="gain")
    imp_parts.append(pd.DataFrame({"fold": f["fold"], "feature": FEATURES, "gain": gain}))
    fit_rows.append({"fold": f["fold"], "best_iteration": best,
                     "capped": best >= CFG.n_estimators,
                     "train_rows": len(tr), "test_days": te.day.nunique()})
    print(f"fold {f['fold']}: best_iter {best:>4} | train {len(tr):>8,} | "
          f"test {te.day.nunique():>4} days")

if not scored_parts:
    raise RuntimeError("no fold produced predictions")
scored = pd.concat(scored_parts, ignore_index=True).sort_values(["day", "security"])
fit_table = pd.DataFrame(fit_rows)
display(fit_table)
if fit_table["capped"].any():
    print("[warn] some folds hit n_estimators without early stopping -- raise n_estimators")

importance = (pd.concat(imp_parts).groupby("feature", as_index=False)["gain"].mean()
              .assign(gain_share=lambda d: d.gain / d.gain.sum())
              .sort_values("gain", ascending=False).reset_index(drop=True))
print("\nfeature importance, averaged over folds:")
display(importance.round(6))

dup = scored.duplicated(["day", "security"]).sum()
print(f"\nout-of-sample rows {len(scored):,} | {scored.day.nunique():,} days | "
      f"duplicated (day, security): {dup}  (must be 0)")


## 6. Evaluate on the test period

In [ ]:
def hac_t(values, lags: int) -> float:
    """Newey-West t-stat of a daily mean. Daily portfolio returns are autocorrelated, so the
    plain t-stat overstates significance."""
    a = pd.to_numeric(values, errors="coerce").to_numpy(dtype=float)
    a = a[np.isfinite(a)]
    n = len(a)
    if n < 2:
        return math.nan
    e = a - a.mean()
    lrv = float(e @ e / n)
    for L in range(1, min(lags, n - 1) + 1):
        lrv += 2.0 * (1.0 - L / (lags + 1.0)) * float(e[L:] @ e[:-L] / n)
    return float(a.mean() / math.sqrt(lrv / n)) if lrv > 0 else math.nan


def max_drawdown(r) -> float:
    d = pd.to_numeric(r, errors="coerce"); d = d[np.isfinite(d)]
    if d.empty or (d < -1.0).any():
        return math.nan
    eq = (1.0 + d).cumprod().to_numpy()
    return float(np.min(eq / np.maximum.accumulate(np.r_[1.0, eq])[1:] - 1.0))


def perf(r) -> dict:
    d = pd.to_numeric(r, errors="coerce"); d = d[np.isfinite(d)]
    if d.empty:
        return {"observations": 0, "ann_mean": math.nan, "ann_vol": math.nan,
                "sharpe": math.nan, "max_drawdown": math.nan}
    am = float(d.mean() * 252.0)
    av = float(d.std(ddof=1) * math.sqrt(252.0)) if len(d) > 1 else math.nan
    return {"observations": len(d), "ann_mean": am, "ann_vol": av,
            "sharpe": am / av if av and math.isfinite(av) and av > 0 else math.nan,
            "max_drawdown": max_drawdown(d)}


# `scored` is already the concatenated out-of-sample series from every fold.
rows = []
for day, g in scored.groupby("day", sort=True):
    r = g.sort_values(["score", "security"], ascending=[False, True], kind="stable")
    k = min(CFG.top_k, len(r))
    top, bot = r.head(k), r.tail(k)
    rows.append({"day": day,
                 "rank_ic": r["score"].rank().corr(r["target_return"].rank()),
                 "top_return": float(top.target_return.mean()),
                 "bottom_return": float(bot.target_return.mean()),
                 "eligible": len(r)})
daily = pd.DataFrame(rows)
daily["spread_return"] = daily.top_return - daily.bottom_return

summary = pd.DataFrame([
    {"leg": leg, **perf(daily[col]),
     f"hac{CFG.hac_lags}_t": hac_t(daily[col], CFG.hac_lags)}
    for leg, col in (("top", "top_return"), ("bottom", "bottom_return"),
                     ("spread", "spread_return"))
])
print(f"test dates {daily.day.nunique():,} | mean daily rank-IC "
      f"{daily.rank_ic.mean():+.4f} (t = {hac_t(daily.rank_ic, CFG.hac_lags):+.2f})")
display(summary.round(4))


In [ ]:
# by calendar year -- a spread that lives in one year is not a result
year = pd.DataFrame([
    {"year": int(y), "start": g.day.min().date(), "end": g.day.max().date(),
     "days": len(g), "rank_ic": g.rank_ic.mean(),
     "top_ann": perf(g.top_return)["ann_mean"], "bottom_ann": perf(g.bottom_return)["ann_mean"],
     "spread_ann": perf(g.spread_return)["ann_mean"],
     "spread_sharpe": perf(g.spread_return)["sharpe"],
     f"spread_hac{CFG.hac_lags}_t": hac_t(g.spread_return, CFG.hac_lags)}
    for y, g in daily.groupby(daily.day.dt.year, sort=True)
])
display(year.round(4))


In [ ]:
# What does the model actually pick? market_cap and dollar volume are NOT features, so this
# is a genuine out-of-sample description of the selection, not a restatement of the inputs.
rows = []
for day, g in scored.groupby("day", sort=True):
    r = g.sort_values(["score", "security"], ascending=[False, True], kind="stable")
    k = min(CFG.top_k, len(r))
    for leg, sel in (("top", r.head(k)), ("universe", r), ("bottom", r.tail(k))):
        rows.append({"leg": leg, "day": day,
                     "median_dollar_volume_21d": float(sel.DOLLARVOLUME_AVG21D.median()),
                     "median_market_cap": float(sel.market_cap.median()),
                     "median_turnover_21day": float(sel.turnover_21day.median()),
                     "zero_volume_share": float(sel.DOLLARVOLUME_AVG21D.eq(0).mean())})
characteristics = (pd.DataFrame(rows).groupby("leg", as_index=False)
                   .mean(numeric_only=True))
display(characteristics.round(4))
print("If top/bottom sit far below the universe on dollar volume or market cap, the spread is")
print("being earned in names where it is hardest to trade -- costs will not be neutral.")


## 7. Cumulative returns

In [ ]:
p = daily.sort_values("day")
fig, ax = plt.subplots(figsize=(11, 5.5))
for col, label, lw in (("top_return", f"Top-{CFG.top_k}", 1.8),
                       ("bottom_return", f"Bottom-{CFG.top_k}", 1.8),
                       ("spread_return", "Top minus Bottom", 2.3)):
    ax.plot(p.day, 100 * p[col].cumsum(), label=label, linewidth=lw)
ax.axhline(0, color="black", linewidth=0.8, alpha=0.6)
ax.set_title("LightGBM-Huber, rv_plus_turnover: cumulative arithmetic daily return")
ax.set_ylabel("cumulative return (pp)"); ax.set_xlabel("test date")
ax.grid(alpha=0.25); ax.legend(); fig.tight_layout(); plt.show()


## 8. Save

In [ ]:
CFG.output_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(CFG.output_dir / "summary.csv", index=False)
year.to_csv(CFG.output_dir / "year_table.csv", index=False)
importance.to_csv(CFG.output_dir / "feature_importance.csv", index=False)
characteristics.to_csv(CFG.output_dir / "selection_characteristics.csv", index=False)
daily.to_parquet(CFG.output_dir / "daily_portfolios.parquet", index=False)

fold_table.to_csv(CFG.output_dir / "folds.csv", index=False)
fit_table.to_csv(CFG.output_dir / "fit_table.csv", index=False)
if CFG.save_model:
    for k, m in models.items():
        m.booster_.save_model(str(CFG.output_dir / f"model_fold{k}.txt"),
                              num_iteration=int(m.best_iteration_ or m.n_estimators))
if CFG.save_row_predictions:
    scored.to_parquet(CFG.output_dir / "row_level_predictions.parquet", index=False)

run = {**{k: str(v) for k, v in asdict(CFG).items()},
       "features": FEATURES, "folds": len(FOLDS),
       "data_rows": len(frame),
       "data_span": [str(frame.day.min().date()), str(frame.day.max().date())],
       "test_dates": int(daily.day.nunique()),
       "mean_rank_ic": float(daily.rank_ic.mean()),
       "python": platform.python_version(), "lightgbm": lgb.__version__,
       "pandas": pd.__version__, "numpy": np.__version__}
(CFG.output_dir / "run_config.json").write_text(json.dumps(run, indent=2))
print("saved to", CFG.output_dir.resolve())


## Reading it

1. **The alignment audit first.** `TRET_T1D == return_1d` must be ~100%. The shortfall on
   `target_return(previous)` is exactly the non-adjacent dates, which are deliberately left
   unlabelled rather than filled from further ahead.
2. **Rank-IC before the portfolio.** It uses the whole cross-section; the top/bottom legs use
   `top_k` names each and are far noisier.
3. **Top and bottom separately.** A spread built entirely by the short leg is a different claim
   from one built by the long leg, and only one of them is easy to trade.
4. **HAC(5), not the raw t-stat.** Daily portfolio returns are autocorrelated and the plain
   t-stat overstates significance.
5. **The year table.** A spread concentrated in one year is not a result.
6. **Selection characteristics.** `market_cap` and dollar volume are not features here, so if
   the legs drift small and illiquid the model found that on its own — and gross returns will
   overstate what is reachable.

Gross predictive results. No trading costs, no borrow, no capacity.
